In [0]:
bronze_batch_df = (
    spark.read
    .table("catalog_smartfactory.bronze.iot_telemetry")
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

silver_batch_df = (
    bronze_batch_df
    .filter(col("machine_id").isNotNull())
    .filter(col("temperature").between(0,200))
    .filter(col("vibration").between(0,20))
    .filter(col("pressure").between(0,300))
    .withColumn("temperature_alerts", when(col("temperature") > 100, 1).otherwise(0))
    .withColumn("vibration_alerts", when(col("vibration") > 4, 1).otherwise(0))
    .withColumn("pressure_alerts", when(col("pressure") > 130, 1).otherwise(0))
    .withColumn(
        "health_score",
        round(
            lit(100)
            - col("temperature_alerts")*20
            - col("vibration_alerts")*30
            - col("pressure_alerts")*20
            - col("failure_risk_score")*30
            ,2)
    )
    .withColumn(
        "machine_status",
        when(col("health_score") >= 90,"HEALTHY")
        .when(col("health_score") >= 70,"WARNING")
        .otherwise("CRITICAL")
    )
)

In [0]:
silver_query = (
    silver_batch_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("catalog_smartfactory.silver.batch_iot_telemetry")

)